### Fetch Hard Graph Problems

In [1]:
%%bash
apt-get update && apt-get install -y zstd git
[ -d NLGraph ] || git clone https://github.com/Arthur-Heng/NLGraph.git
echo "Repository ready."

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [79.0 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,060 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driver

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into 'NLGraph'...
Updating files: 100% (26484/26484), done.


In [2]:
import os, sys, json, re, time, io
import networkx as nx


GRAPH_DIR = "NLGraph/NLGraph/connectivity/graph/hard/standard/"

MAX_PROBLEMS: int = 10

graph_files = sorted([f for f in os.listdir(GRAPH_DIR) if f.endswith(".txt")])
print(f"Found {len(graph_files)} graph files")

all_instances = []

for fname in graph_files:
    file_path = os.path.join(GRAPH_DIR, fname)
    with open(file_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    parts = lines[0].split()
    n = int(parts[0])
    m = int(parts[1])

    edges = []
    idx = 1
    for _ in range(m):
        u, v = map(int, lines[idx].split())
        edges.append((u, v))
        idx += 1

    G = nx.Graph()
    G.add_nodes_from(range(n))
    G.add_edges_from(edges)

    while idx < len(lines):
        u, v = map(int, lines[idx].split())
        idx += 1
        connected = nx.has_path(G, u, v)
        answer = "Yes" if connected else "No"
        all_instances.append({
            "nodes": list(range(n)),
            "edges": edges,
            "question": f"Is node {u} connected to node {v}?",
            "answer": answer,
            "source": u,
            "target": v
        })

if (MAX_PROBLEMS > 0) and (len(all_instances) > MAX_PROBLEMS):
    all_instances = all_instances[0:MAX_PROBLEMS]

print(f"Total test instances: {len(all_instances)}")
print("Example:", all_instances[0])

Found 68 graph files
Total test instances: 10
Example: {'nodes': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25], 'edges': [(0, 9), (1, 18), (1, 8), (1, 4), (2, 15), (2, 18), (2, 11), (2, 23), (2, 4), (3, 12), (3, 5), (4, 23), (4, 25), (5, 12), (5, 17), (6, 22), (6, 13), (7, 18), (7, 23), (7, 8), (9, 17), (10, 19), (10, 17), (11, 15), (12, 24), (14, 20), (14, 22), (15, 18), (16, 20), (16, 22), (20, 22), (21, 22), (23, 25)], 'question': 'Is node 12 connected to node 9?', 'answer': 'Yes', 'source': 12, 'target': 9}


### Fetch LLM Model

In [3]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 13.5 MB/s  0:00:00


In [4]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [5]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

# 1. Load the base Gemma 4 model
model_id = "arnir0/Tiny-LLM" #4B-it" # or E2B, 31B depending on your VRAM
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{ '<|im_start|>assistant\n' }}"
        "{% endif %}"
    )
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], #"all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True,
)

llm_model = get_peft_model(model, peft_config)
llm_model.print_trainable_parameters()
llm_tokenizer = tokenizer

training_config = SFTConfig(
    output_dir="./tinyllm-dora-output",
    per_device_train_batch_size=2,
    packing=True,  #True
    max_length=4096,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    num_train_epochs=3,
    save_strategy="no", #"epoch"
)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


config.json:   0%|          | 0.00/602 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 26.0MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/12 [00:00<?, ?it/s]

trainable params: 11,040 || all params: 13,000,032 || trainable%: 0.0849


In [6]:
USE_GPU: bool = True

device = "cuda" if torch.cuda.is_available() and USE_GPU else "cpu"

### Supervised Finetuning Loop Using Dataset Created by Prop-Opp Pair

In [7]:
!pip install ollama

In [8]:
%%bash
export PATH="/usr/local/bin:$PATH"
curl -fsSL https://ollama.com/install.sh | sh
nohup ollama serve > /tmp/ollama.log 2>&1 &
for i in $(seq 1 30); do
    if curl -s http://localhost:11434 > /dev/null; then
        echo "Ollama ready"
        break
    fi
    sleep 1
done
#ollama run hf.co/mradermacher/Tiny-LLM-GGUF:Q4_K_M
ollama pull gemma4:e2b

Ollama ready


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 6dd7ac24fe13:   0% ▕                  ▏  60 KB/3.5 GB                  pulling manifest 
pulling 6dd7ac24fe13:   0% ▕                  ▏ 8.0 MB/3.5 GB                  pulling manifest 
pulling 6dd7ac24fe13:   3% ▕                  ▏  94 MB/3.5 GB                  pulling manifest 
pulling 6dd7ac24fe13:   6% ▕█                 ▏ 194 MB/3.5 GB                  pulling manifest 

In [9]:
import ollama
from datasets import Dataset
import copy
from transformers import AutoModelForSequenceClassification
import pandas as pd

DATASET_TURNS = 1
N_PASSES = 1
N_INNER_EPOCHS = 5

class FineTuner:
    def __init__(self, 
                 all_problem_instances: list[dict], 
                 llm_tokenizer: torch.nn.Module,
                 llm_model: torch.nn.Module,
                 training_config: SFTConfig,
                 device:torch.device = device,
                 similarity_threshold: float=0.6,
                ) -> None:
        self.all_problem_instances = all_problem_instances
        self.training_config = training_config
        self.device = device
        
        self.llm_model = llm_model
        self.llm_tokenizer = llm_tokenizer
        self.similarity_threshold = similarity_threshold
        
        self.final_proponent_responses = []
        
        # Load a RoBERTa model fine-tuned for semantic similarity / NLI
        model_name = "roberta-large-mnli"
        self.sim_tokenizer = AutoTokenizer.from_pretrained(model_name)
        # model for similarity score
        self.sim_model = AutoModelForSequenceClassification.from_pretrained(model_name)
        
    @torch.no_grad()
    def check_similarity(self, model_answer: str, response: str) -> float:
        inputs = self.sim_tokenizer(model_answer, response, return_tensors="pt", truncation=True, max_length=512)
        logits = self.sim_model(**inputs).logits
        # Convert logits to a probability/similarity score
        probs = torch.softmax(logits, dim=1)
        return probs.tolist()[0][2]

    def create_prompt_for_proponent(self, index: int) -> str:
        nodes_list = ','.join(map(str, self.all_problem_instances[index]["nodes"]))
        edges_list = '[' + ",".join(map(str, self.all_problem_instances[index]["edges"])) + ']'
        prompt = f"""You are a precise graph reasoning agent. Solve the following graph problem. 
        Nodes: {nodes_list}
        Edges: {edges_list}
        Question: {all_instances[index]["question"]}
        
        **Instructions – follow exactly:**
        1. Provide reasoning, with sample paths.
        2. Final line must contain an answer ** Yes/No **
        """
        return prompt

    def create_prompt_for_opponent(self, response: str, index: int) -> str:
        nodes_list = ','.join(map(str, self.all_problem_instances[index]["nodes"]))
        edges_list = '[' + ",".join(map(str, self.all_problem_instances[index]["edges"])) + ']'
        prompt = f"""You are a precise graph reasoning agent that can find errors in reasoning. Examine the following solution, find potential problems and verify if it is correct or not. 
        Nodes: {nodes_list}
        Edges: {edges_list}
        Question: {all_instances[index]["question"]}
        Response: {response}
        
        **Instructions – follow exactly:**
        1. Identify what can go wrong in the ** Response **.
        2. Verify the solution.
        3. Final line must contain an answer ** Yes/No ** indicating if the response is correct solution.
        """
        return prompt
        
    def create_prompt_3_for_proponent(self, opponent_response: str, index: int) -> str:
        nodes_list = ','.join(map(str, self.all_problem_instances[index]["nodes"]))
        edges_list = '[' + ",".join(map(str, self.all_problem_instances[index]["edges"])) + ']'
        clean_response = opponent_response[0:-2] if opponent_response.endswith("No") else opponent_response[0:-3]
        prompt = f"""You are a precise graph reasoning agent. Solve the following graph problem using the provided hint. 
        Nodes: {nodes_list}
        Edges: {edges_list}
        Question: {all_instances[index]["question"]}
        Hint: {clean_response}
        
        **Instructions – follow exactly:**
        1. Provide reasoning, with sample paths.
        2. Final line must contain an answer ** Yes/No **
        """
        return prompt

    def get_responses_using_ollama(self, formatted_prompts: list[str]) -> list[str]:
        result = []
        for prompt in formatted_prompts:
            try:
                response = ollama.generate(
                    model='gemma4:e2b', #"hf.co/mradermacher/Tiny-LLM-GGUF:Q4_K_M", #'arnir0/Tiny-LLM',
                    prompt=prompt
                )
                result.append(response['response'])
            except Exception as e:
                print(e)
                result.append("Error")
            print(response['response'])
        return result
    
    @torch.no_grad()
    def get_level1_proponent_response(self) -> list[str]:
        formatted_prompts = []
        unformatted_prompts = []
        for i in range(len(self.all_problem_instances)):
            llm_prompt = self.create_prompt_for_proponent(i)
            chat = [{"role": "user", "content": llm_prompt}]
            # apply_chat_template adds the correct start/end tokens (<start_of_turn>, <end_of_turn>)
            prompt = self.llm_tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            formatted_prompts.append(prompt)
            unformatted_prompts.append(llm_prompt)

        return self.get_responses_using_ollama(unformatted_prompts)

    @torch.no_grad()
    def get_level2_opponent_response(self, proponent_responses: list[str]) -> list[str]:
        formatted_prompts = []
        unformatted_prompts = []
        for i in range(len(self.all_problem_instances)):
            llm_prompt = self.create_prompt_for_opponent(proponent_responses[i], i)
            chat = [{"role": "user", "content": llm_prompt}]
            # apply_chat_template adds the correct start/end tokens (<start_of_turn>, <end_of_turn>)
            prompt = self.llm_tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            formatted_prompts.append(prompt)
            unformatted_prompts.append(llm_prompt)

        return self.get_responses_using_ollama(unformatted_prompts)
        
    @torch.no_grad()
    def get_level3_proponent_response(self, opponent_responses: list[str]) -> list[str]:
        formatted_prompts = []
        unformatted_prompts = []
        for i in range(len(self.all_problem_instances)):
            llm_prompt = self.create_prompt_3_for_proponent(opponent_responses[i], i)
            chat = [{"role": "user", "content": llm_prompt}]
            # apply_chat_template adds the correct start/end tokens (<start_of_turn>, <end_of_turn>)
            prompt = self.llm_tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            formatted_prompts.append(prompt)
            unformatted_prompts.append(llm_prompt)

        return self.get_responses_using_ollama(unformatted_prompts)
      
    @torch.no_grad()
    def create_dataset(self, improved_responses: list[str], indices: list[int]) -> Dataset:
        prompts = []
        for idx in indices:
            prompt = self.create_prompt_for_proponent(idx)
            prompts.append(prompt)
            
        formatted_list = []
        for p, r in zip(prompts, improved_responses):
            text_block = f"<start_of_turn>user\n{p}<end_of_turn>\n<start_of_turn>model\n{r}<end_of_turn>"
            formatted_list.append(text_block)

        df = pd.DataFrame(data={"Prompt": prompts, 
                                "Response": improved_responses})
        df.to_csv("responses.csv", index=False)

        dataset = Dataset.from_dict({"text": formatted_list})
        return dataset
        
    @torch.no_grad()
    def get_responses_with_improvement(self, proponent_responses: list[str], proponent_responses_3: list[str]) -> tuple[list[str], list[int]]:
        improved_responses = []
        indices = []
        for index in range(len(proponent_responses)):
            prop_resp1 = proponent_responses[index]
            prop_resp2 = proponent_responses_3[index]
            self.final_proponent_responses.append(prop_resp2)
            if self.check_similarity(prop_resp2, prop_resp1) < self.similarity_threshold:
            #if prop_resp1[-2:] != prop_resp2[-2:]:
                improved_responses.append(prop_resp2)
                indices.append(index)

        return improved_responses, indices
      
    def fine_tune(self,
                  problem_instances: list,
                  n_turns: int=DATASET_TURNS,
              ) -> torch.nn.Module:
        self.llm_model.train()
        for _ in range(n_turns):
            proponent_responses = self.get_level1_proponent_response()
            opponent_responses = self.get_level2_opponent_response(proponent_responses)
            proponent_responses_3 = self.get_level3_proponent_response(opponent_responses)

            # use roberta to check for improved reasoning
            improved_responses, indices = self.get_responses_with_improvement(proponent_responses, proponent_responses_3)

            dataset = self.create_dataset(improved_responses, indices)

            trainer = SFTTrainer(
                model=self.llm_model,
                train_dataset=dataset,
                args=self.training_config,
            )

            trainer.train()
            
            # update
            self.llm_model = self.llm_model.merge_and_unload()

        return self.llm_model

    @torch.no_grad()
    def test_llm(self,
                 llm_model: torch.nn.Module,
                 n_problems: int = 10,
                 prefix: str="improved",
                ) -> pd.DataFrame:
        llm_model = llm_model.to(self.device)
        llm_model.eval()
        vocab_size = llm_model.config.vocab_size
        max_valid_id = vocab_size - 1
        prompts = []
        responses = []
        for idx in range(n_problems):
            prompt = self.create_prompt_for_proponent(idx)
            inputs = self.llm_tokenizer(
                            prompt, 
                            return_tensors="pt",
                        ).to(self.device)
            

            #  Clip the input_ids tensor in-place or before generation
            if "input_ids" in inputs:
                inputs["input_ids"] = torch.clamp(inputs["input_ids"], min=0, max=max_valid_id)
            output_ids = llm_model.generate(
                                **inputs,
                                max_new_tokens=1024,
                                do_sample=False,
                                top_k=50,
                                top_p=0.95,
                                temperature=0.7
                            )
            generated_text = self.llm_tokenizer.decode(output_ids[0], skip_special_tokens=True)
            print(generated_text)
            prompts.append(prompt)
            responses.append(generated_text)
        output_df = pd.DataFrame(data={"Prompt": prompts,
                               f"{prefix}Response": responses})
        output_df.to_csv(f"{prefix}_responses.csv", index=False)
        return output_df
        
    @torch.no_grad()
    def evaluate_improvement(self, df:pd.DataFrame) -> pd.DataFrame:
        df.loc[:, "origScore"] = 0.0
        df.loc[:, "improvedScore"] = 0.0
        for i in range(df.shape[0]):
            model_answer = self.final_proponent_responses[i]
            response = df.loc[i, "origResponse"]
            sc_orig = self.check_similarity(model_answer, response)
            df.loc[i, "origScore"] = sc_orig
            response = df.loc[i, "improvedResponse"]
            sc_imp = self.check_similarity(model_answer, response)
            df.loc[i, "improvedScore"] = sc_imp
        df.to_csv(f"scored_responses.csv", index=False)
        return df

In [10]:
fine_tuner = FineTuner(all_instances, llm_tokenizer, llm_model, training_config, device = device)
orig_df = fine_tuner.test_llm(llm_model, prefix="orig")
fine_tuned_llm = fine_tuner.fine_tune(problem_instances=all_instances)
improved_df = fine_tuner.test_llm(fine_tuned_llm, prefix="improved")
merged_df = pd.merge(orig_df, improved_df, left_index=True, right_index=True, how="inner")
final_df = fine_tuner.evaluate_improvement(merged_df)

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.43GB            

... (6 outputs left out)

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Step,Training Loss


You are a precise graph reasoning agent. Solve the following graph problem. 
        Nodes: 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25
        Edges: [(0, 9),(1, 18),(1, 8),(1, 4),(2, 15),(2, 18),(2, 11),(2, 23),(2, 4),(3, 12),(3, 5),(4, 23),(4, 25),(5, 12),(5, 17),(6, 22),(6, 13),(7, 18),(7, 23),(7, 8),(9, 17),(10, 19),(10, 17),(11, 15),(12, 24),(14, 20),(14, 22),(15, 18),(16, 20),(16, 22),(20, 22),(21, 22),(23, 25)]
        Question: Is node 12 connected to node 9?
        
        **Instructions – follow exactly:**
        1. Provide reasoning, with sample paths.
        2. Final line must contain an answer ** Yes/No **
         20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

In [11]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(final_df[["Prompt_x", "improvedResponse", "improvedScore"]])

,Prompt_x,improvedResponse,improvedScore
0,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.203695
1,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.195184
2,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.129809
3,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.276364
4,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.337679
5,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.029235
6,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.243583
7,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.089095
8,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.203036
9,You are a precise graph reasoning agent. Solve...,You are a precise graph reasoning agent. Solve...,0.294723
